# 🔍 Lab 2 — Pipeline RAG : FAISS & Sentence Transformers
### MSc IAG — Cours 17 · AIvancity · Ali Mokh

Dans ce lab, vous allez construire un pipeline **RAG (Retrieval-Augmented Generation)** complet :

| Étape | Outil | Description |
|-------|-------|-------------|
| 📄 Chargement PDF | LangChain + PyPDF | Lire vos documents |
| ✂️ Chunking | RecursiveCharacterTextSplitter | Découper en segments |
| 🔢 Embedding | Sentence Transformers | Vectoriser le texte |
| 🗄️ Stockage | FAISS | Index vectoriel rapide |
| 🤖 Génération | Gemini Flash | Répondre aux questions |
| 🚀 Amélioration | Query Expansion + HyDE | Récupération multi-étapes |

> **Pré-requis :** Un compte Google et une clé API Gemini (gratuite sur [aistudio.google.com](https://aistudio.google.com))

## 0️⃣ Installation des bibliothèques

> ⏳ Cette cellule prend ~1 minute. Exécutez-la une seule fois.

In [ ]:
!pip install -q \
    sentence-transformers \
    faiss-cpu \
    langchain \
    langchain-community \
    langchain-text-splitters \
    pypdf \
    google-generativeai \
    tqdm

print('✅ Installation terminée')

In [ ]:
import os, json, time, warnings
import numpy as np
from pathlib import Path
from tqdm import tqdm

import google.generativeai as genai
from sentence_transformers import SentenceTransformer
import faiss

from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

warnings.filterwarnings('ignore')
print('✅ Imports OK')

## 1️⃣ Configuration

### Clé API Gemini
1. Allez sur [aistudio.google.com](https://aistudio.google.com/app/apikey)
2. Créez une clé API gratuite
3. Dans Colab : cliquez sur 🔑 **Secrets** (panneau gauche) → Ajoutez `GEMINI_API_KEY`
   - Ou collez-la directement dans la variable ci-dessous

In [ ]:
# Option A — via Secrets Colab (recommandé)
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print('✅ Clé chargée depuis les Secrets Colab')
except Exception:
    # Option B — saisissez directement votre clé
    GEMINI_API_KEY = 'VOTRE_CLE_ICI'  # ← remplacez ici si besoin

genai.configure(api_key=GEMINI_API_KEY)
model_gemini = genai.GenerativeModel('gemini-2.0-flash')

# Test rapide
test = model_gemini.generate_content('Réponds en une phrase : qu\'est-ce que le RAG ?')
print('🤖 Test Gemini :', test.text.strip())

## 2️⃣ Chargement de vos PDFs

Uploadez vos fichiers PDF (cours, articles, documentation…).
Le pipeline RAG répondra ensuite aux questions sur ces documents.

> 💡 Pas de PDF sous la main ? Téléchargez un article Wikipédia en PDF ou utilisez le CM du cours.

In [ ]:
from google.colab import files

PDF_DIR = '/content/pdfs'
os.makedirs(PDF_DIR, exist_ok=True)

print('📂 Dossier PDF :', PDF_DIR)
print('⬆️  Sélectionnez vos fichiers PDF ci-dessous :')

uploaded = files.upload()

for filename, content in uploaded.items():
    dest = f'{PDF_DIR}/{filename}'
    with open(dest, 'wb') as f:
        f.write(content)
    print(f'  ✅ {filename} ({len(content)//1024} KB)')

In [ ]:
# Chargement avec LangChain
loader = DirectoryLoader(
    PDF_DIR,
    glob='*.pdf',
    loader_cls=PyPDFLoader,
    show_progress=True
)
documents = loader.load()

sources = set(d.metadata.get('source', '') for d in documents)
print(f'\n📊 Résultat :')
print(f'   {len(sources)} fichier(s) PDF chargé(s)')
print(f'   {len(documents)} page(s) extraite(s)')
print(f'\n--- Aperçu de la 1ère page ---')
print(documents[0].page_content[:600])

## 3️⃣ Découpage en Chunks

Le **chunking** divise chaque page en petits segments de texte.
Pourquoi ? Les LLMs ont une fenêtre de contexte limitée, et des chunks courts
permettent une récupération plus précise.

| Paramètre | Valeur | Signification |
|-----------|--------|---------------|
| `chunk_size` | 500 | ~500 caractères par chunk |
| `chunk_overlap` | 50 | 50 caractères de chevauchement pour ne pas couper les idées |

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=['\n\n', '\n', '. ', ' ', '']
)

chunks = splitter.split_documents(documents)

sizes = [len(c.page_content) for c in chunks]
print(f'✅ Chunking terminé :')
print(f'   {len(documents)} pages  →  {len(chunks)} chunks')
print(f'   Taille moyenne : {int(np.mean(sizes))} car. | Min : {min(sizes)} | Max : {max(sizes)}')

print(f'\n--- Exemple : chunk n°5 ---')
print(chunks[5].page_content)
print('Métadonnées :', chunks[5].metadata)

## 4️⃣ Embedding avec Sentence Transformers

L'**embedding** transforme chaque chunk en un vecteur numérique.
Des chunks sémantiquement similaires auront des vecteurs proches dans l'espace.

Modèle utilisé : **`all-MiniLM-L6-v2`**
- 384 dimensions
- Rapide, léger, multilingue (fonctionne en français)
- Open-source, gratuit, tourne en local

In [ ]:
print('⏳ Chargement du modèle d\'embedding...')
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print(f'✅ Modèle chargé — {embed_model.get_sentence_embedding_dimension()} dimensions')

# Test sur une phrase
test_vec = embed_model.encode('Qu\'est-ce que le RAG ?')
print(f'📐 Vecteur exemple (5 premières valeurs) : {test_vec[:5].round(4)}')

In [ ]:
print('⏳ Encodage de tous les chunks...')
texts = [c.page_content for c in chunks]
embeddings = embed_model.encode(
    texts,
    show_progress_bar=True,
    batch_size=32
)

print(f'\n✅ {len(embeddings)} vecteurs créés')
print(f'   Dimension : {embeddings.shape[1]}')
print(f'   Taille totale : {embeddings.nbytes // 1024} KB en mémoire')

## 5️⃣ Index Vectoriel FAISS

**FAISS** (Facebook AI Similarity Search) est une bibliothèque ultra-rapide pour
rechercher des vecteurs similaires parmi des millions d'entrées.

On normalise les vecteurs (L2) pour que le produit scalaire équivaille à la **similarité cosinus**.

In [ ]:
# Normalisation L2 → produit scalaire = similarité cosinus
embeddings_f32 = np.array(embeddings).astype('float32')
faiss.normalize_L2(embeddings_f32)

# Création de l'index
dim = embeddings_f32.shape[1]
index = faiss.IndexFlatIP(dim)  # IndexFlatIP = produit scalaire (cosine après normalisation)
index.add(embeddings_f32)

print(f'✅ Index FAISS créé : {index.ntotal} vecteurs')
print(f'   Dimension : {dim}')

In [ ]:
def retrieve(query: str, k: int = 5) -> list:
    """Recherche les k chunks les plus similaires à la requête."""
    query_emb = embed_model.encode([query]).astype('float32')
    faiss.normalize_L2(query_emb)
    scores, indices = index.search(query_emb, k)
    return [
        {'score': float(s), 'text': chunks[i].page_content,
         'source': chunks[i].metadata.get('source', '')}
        for s, i in zip(scores[0], indices[0])
    ]

# Test de recherche
query_test = 'Qu\'est-ce que le RAG ?'  # ← modifiez selon vos PDFs
results = retrieve(query_test, k=3)

print(f'🔍 Requête : "{query_test}"\n')
for i, r in enumerate(results, 1):
    print(f'--- Résultat {i} (score: {r["score"]:.4f}) ---')
    print(r['text'][:300])
    print()

## 6️⃣ Génération avec Gemini — Pipeline RAG Complet

On assemble les 3 étapes en une seule fonction :
1. **Retrieve** → les chunks les plus proches de la question
2. **Augment** → construire un prompt avec ces chunks comme contexte
3. **Generate** → Gemini génère la réponse ancrée dans le contexte

In [ ]:
def rag(question: str, k: int = 5, verbose: bool = True) -> str:
    """Pipeline RAG de base : retrieve → augment → generate."""

    # 1. Retrieve
    retrieved = retrieve(question, k=k)
    context = '\n\n---\n\n'.join(r['text'] for r in retrieved)

    # 2. Augment
    prompt = f"""Tu es un assistant pédagogique expert.
Réponds à la question en te basant UNIQUEMENT sur le contexte fourni ci-dessous.
Si la réponse n'est pas dans le contexte, dis-le clairement.
Réponds en français.

CONTEXTE :
{context}

QUESTION : {question}

RÉPONSE :"""

    # 3. Generate
    response = model_gemini.generate_content(prompt)

    if verbose:
        print(f'📚 {len(retrieved)} chunks utilisés comme contexte :')
        for i, r in enumerate(retrieved, 1):
            print(f'  [{i}] score={r["score"]:.3f} — {r["text"][:80]}...')
        print()

    return response.text

# ── Test ──────────────────────────────────────────────────────────────────────
question = 'Qu\'est-ce que le RAG et pourquoi est-il utile ?'  # ← adaptez à vos PDFs
print('💬 Question :', question)
print()
reponse = rag(question)
print('💡 Réponse :')
print(reponse)

In [ ]:
# 🧪 Testez votre propre question !
ma_question = 'Expliquez le mécanisme principal du document.'  # ← modifiez ici

print(f'💬 Votre question : {ma_question}\n')
print('💡 Réponse RAG :')
print(rag(ma_question, verbose=False))

---
## 7️⃣ Amélioration : Récupération Multi-Étapes

Un RAG basique a des limites :
- Une question vague peut rater des chunks pertinents
- Le vocabulaire de la question peut différer de celui des documents

**Deux techniques pour améliorer la récupération :**

### Technique A — Query Expansion (Expansion de Requête)
Le LLM génère **3 reformulations** de la question → on récupère pour chacune → on fusionne et déduplique.

```
Question originale ──────────────────────────────────► Retrieve ─┐
            └─► LLM → Variante 1 ──────────────────► Retrieve ──┤─► Merge → Generate
                         └─► Variante 2 ────────────► Retrieve ──┘
```

In [ ]:
def expand_query(question: str) -> list:
    """Génère des variantes de la question pour enrichir la récupération."""
    prompt = f"""Génère 3 formulations différentes de cette question pour améliorer la recherche documentaire.
Retourne UNIQUEMENT un JSON valide avec une clé \"queries\".

Question : {question}

Format : {{\"queries\": [\"variante 1\", \"variante 2\", \"variante 3\"]}}"""

    try:
        resp = model_gemini.generate_content(prompt)
        text = resp.text.strip()
        # Nettoyer les blocs de code Markdown si présents
        if '```' in text:
            text = text.split('```')[1].replace('json', '').strip()
        data = json.loads(text)
        return [question] + data.get('queries', [])
    except Exception as e:
        print(f'⚠️ Query expansion échouée ({e}), requête originale utilisée')
        return [question]


def rag_query_expansion(question: str, k: int = 4) -> str:
    """RAG avec expansion de requête."""

    # 1. Générer les variantes
    queries = expand_query(question)
    print(f'🔍 {len(queries)} requêtes générées :')
    for q in queries:
        print(f'   • {q}')

    # 2. Récupérer et dédupliquer
    seen, all_chunks = set(), []
    for q in queries:
        for r in retrieve(q, k=k):
            if r['text'] not in seen:
                seen.add(r['text'])
                all_chunks.append(r)

    # 3. Trier par score, garder les meilleurs
    all_chunks.sort(key=lambda x: x['score'], reverse=True)
    top = all_chunks[:k * 2]
    context = '\n\n---\n\n'.join(r['text'] for r in top)

    print(f'\n📚 {len(top)} chunks uniques conservés (sur {len(all_chunks)} récupérés)\n')

    # 4. Générer
    prompt = f"""Tu es un assistant pédagogique expert.
Réponds à la question en te basant UNIQUEMENT sur le contexte fourni.
Réponds en français.

CONTEXTE :
{context}

QUESTION : {question}

RÉPONSE :"""
    return model_gemini.generate_content(prompt).text


# Test
q = 'Qu\'est-ce que le RAG ?'  # ← adaptez
print('=' * 60)
reponse_qe = rag_query_expansion(q)
print('💡 Réponse (Query Expansion) :')
print(reponse_qe)

### Technique B — HyDE (Hypothetical Document Embeddings)

Au lieu d'embedder la question directement, on demande au LLM de **générer une réponse hypothétique**,
puis on embedde cette réponse pour trouver des chunks qui y ressemblent.

```
Question ──► LLM : génère une réponse hypothétique
                         │
                         ▼
               Embed(réponse hypothétique) ──► FAISS search ──► Generate
```

> 💡 **Pourquoi ça marche ?** La réponse hypothétique utilise le même vocabulaire
> que les documents, contrairement à une question courte et vague.

In [ ]:
def rag_hyde(question: str, k: int = 5) -> str:
    """RAG avec HyDE (Hypothetical Document Embeddings)."""

    # Étape 1 : Générer une réponse hypothétique
    print('💭 Génération de la réponse hypothétique...')
    hyp_prompt = f"""Génère un paragraphe de 3-4 phrases qui POURRAIT être la réponse
à cette question dans un document académique sur l'IA générative.
Génère une réponse plausible et détaillée, même si tu n'es pas certain.

Question : {question}

Réponse hypothétique :"""

    hyp_answer = model_gemini.generate_content(hyp_prompt).text
    print(f'   → {hyp_answer[:180]}...\n')

    # Étape 2 : Embedder la réponse hypothétique
    hyp_emb = embed_model.encode([hyp_answer]).astype('float32')
    faiss.normalize_L2(hyp_emb)

    # Étape 3 : Chercher des chunks similaires à cette réponse
    scores, indices = index.search(hyp_emb, k)
    retrieved = [
        {'score': float(s), 'text': chunks[i].page_content}
        for s, i in zip(scores[0], indices[0])
    ]
    context = '\n\n---\n\n'.join(r['text'] for r in retrieved)
    print(f'📚 {len(retrieved)} chunks récupérés via HyDE\n')

    # Étape 4 : Générer la vraie réponse
    prompt = f"""Tu es un assistant pédagogique expert.
Réponds à la question en te basant UNIQUEMENT sur le contexte fourni.
Réponds en français.

CONTEXTE :
{context}

QUESTION : {question}

RÉPONSE :"""
    return model_gemini.generate_content(prompt).text


# Test
q = 'Qu\'est-ce que le RAG ?'  # ← adaptez
print('=' * 60)
reponse_hyde = rag_hyde(q)
print('💡 Réponse (HyDE) :')
print(reponse_hyde)

### Comparaison des 3 approches

Testez la même question avec les 3 méthodes et observez les différences.

In [ ]:
question = 'Expliquez le mécanisme principal décrit dans le document.'  # ← adaptez

print('━' * 70)
print('APPROCHE 1 — RAG BASIQUE')
print('━' * 70)
r1 = rag(question, verbose=False)
print(r1)

print('\n' + '━' * 70)
print('APPROCHE 2 — QUERY EXPANSION')
print('━' * 70)
r2 = rag_query_expansion(question)
print(r2)

print('\n' + '━' * 70)
print('APPROCHE 3 — HyDE')
print('━' * 70)
r3 = rag_hyde(question)
print(r3)

---
## ✅ Récapitulatif

Vous avez construit un pipeline RAG complet en 6 étapes :

```
PDFs → Chunking → Embedding (MiniLM) → FAISS → Retrieve → Gemini → Réponse
```

Et deux techniques d'amélioration :

| Technique | Principe | Avantage |
|-----------|----------|----------|
| **Query Expansion** | Générer des variantes de la question | Couvre plus de vocabulary |
| **HyDE** | Embedder une réponse hypothétique | Meilleure correspondance sémantique |

**➡️ Prochaine étape : Lab 3** — même pipeline mais avec les services GCP (Vertex AI Embeddings + Gemini pro)

---
# 📝 Exercices — À compléter

> Ces exercices évaluent votre compréhension du pipeline RAG construit dans ce lab.
> Complétez chaque cellule marquée `# TODO` et exécutez-la pour vérifier votre réponse.
> **Rendu :** soumettez ce notebook avec toutes les cellules exécutées.


## Exercice 1 — Impact du Chunking *(2 pts)*

La taille des chunks influence directement la qualité de la récupération.

**Tâche :** Relancez le pipeline avec `chunk_size=200` et `chunk_size=1000`.
Pour chaque configuration, posez la même question et comparez :
- Le nombre de chunks créés
- Les 3 premiers chunks récupérés (texte + score)
- La qualité perçue de la réponse finale

Répondez en commentaire dans la cellule.

In [ ]:
# ── Exercice 1 ───────────────────────────────────────────────────────────
# TODO: Redéfinissez splitter avec chunk_size=200, rechunker, ré-embedder,
#       reconstruire l'index FAISS, poser une question et afficher les résultats.
# Répétez avec chunk_size=1000.

QUESTION_TEST = 'Résumez le sujet principal du document.'

for chunk_size in [200, 1000]:
    print(f'\n{'='*60}')
    print(f'chunk_size = {chunk_size}')
    print('='*60)

    # TODO: re-chunker
    splitter_test = None  # ← remplacez None

    # TODO: re-embedder et reconstruire l'index
    # ...

    # TODO: récupérer les 3 premiers chunks et afficher score + texte
    # ...

    # TODO: générer la réponse et l'afficher
    # ...

# TODO: écrivez ici vos observations (en commentaire)
# Avec chunk_size=200 : ...
# Avec chunk_size=1000 : ...
# Conclusion : ...


## Exercice 2 — Modèle d'Embedding Multilingue *(2 pts)*

Le modèle `all-MiniLM-L6-v2` est optimisé pour l'anglais.
Pour des documents en français, `paraphrase-multilingual-MiniLM-L12-v2` peut être meilleur.

**Tâche :** Remplacez le modèle d'embedding par le modèle multilingue,
reconstruisez l'index, et comparez les scores de récupération sur 2 questions en français.


In [ ]:
# ── Exercice 2 ───────────────────────────────────────────────────────────
QUESTIONS = [
    'Qu\'est-ce que le RAG ?',
    'Quels sont les avantages de cette approche ?',
]

# TODO: chargez le modèle multilingue
# embed_model_ml = SentenceTransformer('...')  # ← complétez le nom du modèle

# TODO: encodez les chunks avec ce nouveau modèle
# embeddings_ml = ...

# TODO: construisez un nouvel index FAISS index_ml
# ...

# TODO: pour chaque question, récupérez les 3 meilleurs chunks
#       avec MiniLM (index original) ET avec le modèle multilingue (index_ml)
#       et affichez les scores côte à côte
for q in QUESTIONS:
    print(f'\nQuestion : {q}')
    print('  [MiniLM]      :', '...')  # TODO: remplacez '...' par les scores réels
    print('  [Multilingue] :', '...')  # TODO

# TODO: commentaire — quel modèle donne de meilleurs scores sur vos PDFs français ?
# Conclusion : ...


## Exercice 3 — Évaluation manuelle du RAG *(3 pts)*

**Tâche :** Définissez 3 paires *(question, réponse attendue)* à partir de vos PDFs.
Pour chaque question, comparez la réponse RAG avec la réponse attendue et attribuez
un score de 0 à 2 selon la grille ci-dessous.

| Score | Critère |
|-------|---------|
| 0 | Réponse incorrecte ou hors sujet |
| 1 | Réponse partiellement correcte |
| 2 | Réponse correcte et complète |

Calculez le score moyen final.

In [ ]:
# ── Exercice 3 ───────────────────────────────────────────────────────────
# TODO: définissez 3 paires (question, réponse attendue) basées sur vos PDFs
qa_pairs = [
    {
        'question': '...',          # TODO: votre question
        'expected': '...',          # TODO: réponse attendue (d'après le PDF)
    },
    {
        'question': '...',
        'expected': '...',
    },
    {
        'question': '...',
        'expected': '...',
    },
]

scores = []
for i, pair in enumerate(qa_pairs, 1):
    print(f'\n--- Question {i} ---')
    print(f'Q : {pair["question"]}')

    # TODO: obtenez la réponse RAG
    # rag_response = rag(pair['question'], verbose=False)
    rag_response = '...'  # ← remplacez
    print(f'RAG    : {rag_response[:200]}')
    print(f'Attendu: {pair["expected"]}')

    # TODO: attribuez un score de 0, 1 ou 2
    score = None  # ← remplacez par 0, 1 ou 2
    scores.append(score)
    print(f'Score  : {score}/2')

# TODO: calculez et affichez le score moyen
# score_moyen = ...
# print(f'\n📊 Score moyen : {score_moyen:.2f}/2')


## Exercice 4 — Votre propre technique Multi-step *(3 pts)*

Vous avez vu deux techniques de récupération multi-étapes : **Query Expansion** et **HyDE**.

**Tâche :** Implémentez une troisième technique de votre choix parmi :

- **Step-Back Prompting** : avant de récupérer, demandez au LLM de reformuler la question
  en une question plus générale ("step back"), puis récupérez pour les deux versions
- **Re-ranking** : récupérez d'abord k=10 chunks, puis demandez au LLM de classer
  les 3 plus pertinents avant de générer
- **Chain-of-Thought RAG** : forcez Gemini à raisonner étape par étape avant de répondre
  en ajoutant `Raisonne étape par étape :` dans le prompt

Comparez votre technique avec le RAG basique sur au moins 2 questions.

In [ ]:
# ── Exercice 4 ───────────────────────────────────────────────────────────
# TODO: choisissez une technique et implémentez-la
# Technique choisie : ...  (Step-Back / Re-ranking / CoT-RAG)

def rag_custom(question: str, k: int = 5) -> str:
    """
    TODO: décrivez votre technique en quelques lignes ici.
    """
    # TODO: implémentez votre logique ici
    pass


# TODO: testez sur 2 questions et comparez avec rag()
questions = [
    '...',  # TODO: votre question 1
    '...',  # TODO: votre question 2
]

for q in questions:
    print(f'\n{'='*60}')
    print(f'Question : {q}')
    print('\n[RAG basique]')
    print(rag(q, verbose=False))
    print('\n[Votre technique]')
    # print(rag_custom(q))  # ← décommentez quand prêt

# TODO: conclusion — votre technique est-elle meilleure ? Pourquoi ?
# Conclusion : ...
